In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('data/processed/dataset_sucio.csv')

In [3]:
# limpiamos los nulos, duplicados y columnas innecesarias

In [4]:
eliminar = []

for col in df.columns:
    if df[col].isna().sum() > len(df) * 0.15:
        eliminar.append(col)

print(eliminar)


['pobr', 'fum', 'alc', 'obes', 'fyv', 'sati']


In [5]:
df = df.drop(columns= eliminar)

Las variables 'crim' y 'tasa_criminalidad' son iguales... en 'crim' tenemos mas nulos por que no tenemos datos de Castilla Y León, esto seguramente se debe a una perdida de datos a la hora de formar el dataset.  
Las varibles 'precio_medio_anual_eur_m2_venta','precio_medio_anual_eur_m2_alquiler' faltan datos de algunos años de Ceuta, Melilla, Navarra, País Vasco y Rioja. No parece ser una perdida de datos si no a la no existencia de los mismos.

In [6]:
df = df.drop(columns= ['crim'])

In [7]:
col_con_nulos = ['homi','tasa_criminalidad', 'precio_medio_anual_eur_m2_venta','precio_medio_anual_eur_m2_alquiler'] 

In [8]:
for col in col_con_nulos:
    bfill = df.groupby("com_aut")[col].bfill()
    mask = (df["año"] == 2009) & (df[col].isna())
    df.loc[mask, col] = bfill[mask]

In [9]:
cols_precios = ["precio_medio_anual_eur_m2_venta", "precio_medio_anual_eur_m2_alquiler"]
df = df.sort_values(["com_aut", "año"]).copy()

df[cols_precios] = (
    df.groupby("com_aut")[cols_precios]
      .transform(lambda x: x.interpolate(method="linear", limit_area="inside"))
)

In [10]:
df = df.sort_values(["com_aut", "año"]).copy()

df[cols_precios] = df.groupby("com_aut")[cols_precios].ffill()

In [11]:
df[df.duplicated()]

,año,com_aut,pib,pob,sui,nat,paro,ing,homi,pib_pc,...,retrasos_pagos(%),renta_media,renta_mediana,riesgo_pobreza(%),dificultad_fin_mes(%),desigualdad_ing(S80/S20),inc_gastos_imprevistos(%),tasa_criminalidad,precio_medio_anual_eur_m2_venta,precio_medio_anual_eur_m2_alquiler


In [12]:
df.isna().sum()

año                                    0
com_aut                                0
pib                                    0
pob                                    0
sui                                    0
nat                                    0
paro                                   0
ing                                    0
homi                                  14
pib_pc                                 0
renta_pc                               0
poblacion                              0
gasto_elevado_vivienda(%)              0
falta_espacio_vivienda(%)              0
retrasos_pagos(%)                      0
renta_media                            0
renta_mediana                          0
riesgo_pobreza(%)                      0
dificultad_fin_mes(%)                  0
desigualdad_ing(S80/S20)               0
inc_gastos_imprevistos(%)              0
tasa_criminalidad                      0
precio_medio_anual_eur_m2_venta        0
precio_medio_anual_eur_m2_alquiler     0
dtype: int64

In [74]:
df.to_csv("data/processed/dataset_final.csv", index=False)

### Resumen del proceso de limpieza del dataset

En primer lugar, se realizó un análisis de valores ausentes por columna. A partir de este análisis, se decidió eliminar aquellas columnas que presentaban más del 15 % de valores nulos respecto al total de filas del dataset. Para ello, se calculó el umbral multiplicando el número total de filas por 0,15, y se eliminaron todas las columnas cuyo número de valores ausentes superaba dicho umbral.

En una segunda etapa, se identificaron variables redundantes que representaban la misma información. En estos casos, se conservó una única variable y se eliminó la columna duplicada para evitar introducir ruido o duplicación innecesaria en el análisis.

Posteriormente, se abordó el tratamiento de variables que presentaban valores ausentes concentrados principalmente en el año inicial del dataset (2009). Dado que el conjunto de datos comienza en ese año y que para algunas comunidades no existían datos oficiales en 2009 pero sí en 2010, se optó por imputar dichos valores utilizando el dato correspondiente al año 2010 dentro de la misma comunidad autónoma, preservando así la continuidad temporal sin eliminar el año completo.

Por último, para las variables de precio medio anual de vivienda (venta y alquiler), se aplicó una estrategia de imputación en dos pasos. En primer lugar, se realizó una interpolación temporal lineal intra-comunidad para completar huecos intermedios entre años con datos disponibles. En segundo lugar, para los valores ausentes que permanecían en los años finales de la serie (al no existir un año posterior que permitiera la interpolación), se utilizó un forward fill intra-comunidad, imputando dichos valores con el último dato disponible para la misma comunidad. Este procedimiento se limitó exclusivamente a las variables de precios y se consideró una aproximación conservadora.
